In [1]:
# Implementing Attention in JAX: the code from the post.
# pip install jax flax optax
# versions used: jax 0.10.2, flax 0.12.8, optax 0.2.8

In [2]:
import jax
import jax.numpy as jnp
from flax import nnx


class Attention(nnx.Module):
    """Single-head attention: three bias-free projections and two matrix multiplies."""

    def __init__(self, d_model: int, d_k: int, d_v: int, *, rngs: nnx.Rngs):
        # use_bias=False makes each projection exactly x @ W, with no affine offset
        self.w_q = nnx.Linear(d_model, d_k, use_bias=False, rngs=rngs)
        self.w_k = nnx.Linear(d_model, d_k, use_bias=False, rngs=rngs)
        self.w_v = nnx.Linear(d_model, d_v, use_bias=False, rngs=rngs)

    def __call__(self, x_query, x_context):
        queries = self.w_q(x_query)      # (n, d_k)
        keys = self.w_k(x_context)       # (m, d_k)
        values = self.w_v(x_context)     # (m, d_v)

        scores = queries @ keys.T                    # (n, m), one score per pair
        weights = jax.nn.softmax(scores, axis=-1)    # each row sums to 1
        return weights @ values                      # (n, d_v)

In [3]:
model = Attention(d_model=16, d_k=8, d_v=12, rngs=nnx.Rngs(0))
context = jnp.ones((7, 16))
sequence = jnp.ones((5, 16))

cross = model(sequence, context)   # (5, 12), cross-attention
self_attended = model(sequence, sequence)   # (5, 12), self-attention

print(cross.shape, self_attended.shape)

(5, 12) (5, 12)


In [4]:
# the softmax normalizes along the key axis, so every query's weights sum to 1
queries = model.w_q(sequence)
keys = model.w_k(context)
weights = jax.nn.softmax(queries @ keys.T, axis=-1)

print(weights.shape, weights.sum(axis=-1))

(5, 7) [1. 1. 1. 1. 1.]


In [5]:
import optax

optimizer = nnx.Optimizer(model, optax.adamw(1e-3), wrt=nnx.Param)


def loss_fn(model, x, context, target):
    return jnp.mean((model(x, context) - target) ** 2)


target = jnp.zeros((5, 12))   # stand-in for whatever the task supervises

loss, grads = nnx.value_and_grad(loss_fn)(model, sequence, context, target)
optimizer.update(model, grads)

In [6]:
# every gradient has the shape of the weight matrix it belongs to
for name in ['w_q', 'w_k', 'w_v']:
    print(name, getattr(grads, name).kernel.get_value().shape)

print('loss', loss)

w_q (16, 8)
w_k (16, 8)
w_v (16, 12)
loss 1.0265075
